# pysystemtrade walkthrough — match the Excel model to the real library

This notebook reproduces, line by line, what the Excel model in `pysystemtrade_learning_model.xlsx` does — but using the actual `pysystemtrade` library rather than hand-rolled formulas. The point is to give you a Rosetta Stone: every Excel cell maps to a function call here.

**Pipeline being demonstrated** (`systems/provided/futures_chapter15/basesystem.py`):

    RawData → Rules → ForecastScaleCap → ForecastCombine → PositionSizing → Portfolio → Account

**How to run.** From the repo root:
```bash
python -m pip install --editable '.[dev]'
jupyter notebook learning/walkthrough.ipynb
```

If you don't have IB or a private config, the library falls back to bundled CSV sample data — perfect for learning.

## 1. Load the chapter-15 system

This is the same system used in Carver's book and the Excel model. It wires together the `RawData`, `Rules`, `ForecastScaleCap`, `ForecastCombine`, `PositionSizing`, `Portfolios`, and `Account` stages with the chapter-15 config.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from systems.provided.futures_chapter15.basesystem import futures_system
system = futures_system()
INSTR = 'GOLD'
print('Stages in pipeline:')
for s in system.stage_names:
    print('   -', s)


## 2. Raw data — the 'Prices' sheet equivalent

Excel `Prices!C` (price), `Prices!D` (return), `Prices!G` (blended daily vol).

Library calls:
- `system.rawdata.get_daily_prices(INSTR)`
- `system.rawdata.daily_returns(INSTR)`
- `system.rawdata.daily_returns_volatility(INSTR)` — uses `mixed_vol_calc` (35d EWM blended 70/30 with 10yr long-term EWM, see `sysdata/config/defaults.yaml` lines 28-37).

In [ ]:
price = system.rawdata.get_daily_prices(INSTR)
rets  = system.rawdata.daily_returns(INSTR)
vol   = system.rawdata.daily_returns_volatility(INSTR)        # daily price-level vol
perc_vol = system.rawdata.get_daily_percentage_volatility(INSTR)  # daily % vol — matches Excel Prices!G
df = pd.DataFrame({'price': price, 'return': rets, 'price_vol': vol, 'perc_vol': perc_vol})
df.tail()

## 3. Trading rule forecasts — the 'Rules' sheet

Excel `Rules!I` is the capped EWMAC 64/256 forecast, `Rules!M` is EWMAC 16/64, `Rules!Q` is carry, `Rules!R` is breakout.

Library calls:
- `system.rules.get_raw_forecast(INSTR, 'ewmac64_256')` — output of `systems/provided/rules/ewmac.py:ewmac()`. Formula: `(price.ewm(64).mean() - price.ewm(256).mean()) / vol`.
- `system.forecastScaleCap.get_capped_forecast(INSTR, 'ewmac64_256')` — multiplied by scalar 1.87 (chapter15 config) and clipped to ±20.


In [ ]:
rules_in_play = list(system.rules.trading_rules().keys())
print('Active rules:', rules_in_play)

fc = pd.DataFrame({
    rule: system.forecastScaleCap.get_capped_forecast(INSTR, rule)
    for rule in rules_in_play
})
fc.tail()


In [ ]:
# Compare a single rule end-to-end so the Excel formulas can be verified.
rule = 'ewmac64_256'
raw    = system.rules.get_raw_forecast(INSTR, rule)               # Excel: Rules!G
scaled = system.forecastScaleCap.get_scaled_forecast(INSTR, rule) # Excel: Rules!H
capped = system.forecastScaleCap.get_capped_forecast(INSTR, rule) # Excel: Rules!I
compare = pd.concat([raw.rename('raw'), scaled.rename('scaled'), capped.rename('capped')], axis=1)
compare.tail(10)

## 4. Combined forecast — the 'Combine' sheet

Excel `Combine!I` = weighted sum × FDM, then capped.

Library: `systems/forecast_combine.py`. The combined forecast equals
`Σ(weight_i × forecast_i) × FDM`, capped at ±20.

In [ ]:
weights = system.combForecast.get_forecast_weights(INSTR)
fdm     = system.combForecast.get_forecast_diversification_multiplier(INSTR)
combined= system.combForecast.get_combined_forecast(INSTR)
print('Forecast weights (last row):')
print(weights.iloc[-1])
print('\nFDM (last value):', fdm.iloc[-1])
print('\nCombined forecast (tail):')
combined.tail()

## 5. Position sizing — the 'Position' sheet

Excel `Position!J` = subsystem position = vol_scalar × forecast / 10.

Library chain (`systems/positionsizing.py`):
- `get_daily_cash_vol_target()` — `capital × vol% / sqrt(256)` → matches Excel `Position!H`.
- `get_instrument_value_vol(INSTR)` — `price × point_size × FX × daily_perc_vol` → matches Excel `Position!G`.
- `get_volatility_scalar(INSTR)` — ratio → matches Excel `Position!I`.
- `get_subsystem_position(INSTR)` — `vol_scalar × forecast / 10` → matches Excel `Position!J`.

In [ ]:
daily_cash_vt = system.positionSize.get_daily_cash_vol_target()
iv_vol        = system.positionSize.get_instrument_value_vol(INSTR)
vol_scalar    = system.positionSize.get_volatility_scalar(INSTR)
sub_pos       = system.positionSize.get_subsystem_position(INSTR)
print('Daily cash vol target:', daily_cash_vt)
print('\nLast 5 days:')
pd.DataFrame({
    'instr_value_vol': iv_vol,
    'vol_scalar':      vol_scalar,
    'subsystem_pos':   sub_pos,
}).tail()

## 6. Portfolio — the 'Portfolio' sheet

Excel `Portfolio!L` = subsystem × inst_weight × IDM.

Library: `systems/portfolio.py`. `get_notional_position(INSTR)` returns the final, instrument-weighted, IDM-multiplied position.

In [ ]:
iw  = system.portfolio.get_instrument_weights()
idm = system.portfolio.get_instrument_diversification_multiplier()
print('Instrument weights (last row):')
print(iw.iloc[-1])
print('\nIDM (last value):', idm.iloc[-1])

for inst in iw.columns[:6]:
    pos = system.portfolio.get_notional_position(inst)
    print(f'{inst:<10} final position (last day): {pos.iloc[-1]:>8.2f}')

## 7. Optional — sanity-check Excel against library numerically

If you copy the same prices the library produces into the Excel `Prices!C` column, the EWMAC forecast values should match the library's `get_capped_forecast` to within a fraction of a unit.

The reason for any small drift is that Excel's `STDEV` over a window is a simple-windowed stdev, while pysystemtrade uses an exponentially-weighted stdev (`ewm(span=35).std()`). For the larger pipeline this is a sub-1% effect.

In [ ]:
# Verify: hand-compute EWMAC 64/256 the way Excel does, compare to library output.
fast = price.ewm(span=64, min_periods=1).mean()
slow = price.ewm(span=256, min_periods=1).mean()
raw_hand = (fast - slow) / vol
raw_lib  = system.rules.get_raw_forecast(INSTR, 'ewmac64_256')
diff = (raw_hand - raw_lib).dropna()
print('Hand-rolled vs library raw EWMAC: max abs diff =', diff.abs().max())
print('Mean abs diff:', diff.abs().mean())

## 8. Plot — see what the system actually does

Forecast vs price — when the trend is strong, forecast pegs at +20 (or -20); position size also scales up.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
price.tail(500).plot(ax=axes[0], title=f'{INSTR} price'); axes[0].grid(True, alpha=.3)
combined.tail(500).plot(ax=axes[1], title='Combined forecast (capped at \u00b120)')
axes[1].axhline(20, linestyle='--', alpha=.4); axes[1].axhline(-20, linestyle='--', alpha=.4)
axes[1].axhline(0, color='k', linewidth=.6); axes[1].grid(True, alpha=.3)
system.portfolio.get_notional_position(INSTR).tail(500).plot(ax=axes[2], title='Notional position (contracts)')
axes[2].axhline(0, color='k', linewidth=.6); axes[2].grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## 9. Discretionary takeaway

Three insights worth internalising:

1. **Forecast normalisation.** Every signal is forced onto the same `±20` scale so they are comparable and combinable. For a discretionary trader, the equivalent is having a single 'conviction score' you use for every trade — not 'super high', 'pretty good', 'meh'. A consistent scale is what lets you size consistently.
2. **Vol scalar is everything.** The reason Carver-style sizing works in any market is: same dollar risk = different contract count depending on volatility. A 6-conviction trade in a 40%-vol name is HALF the contracts of a 6-conviction trade in a 20%-vol name. If you don't adjust contract count when vol changes, your portfolio risk drifts.
3. **IDM is the diversification dividend.** Holding 10 uncorrelated bets at vol_target/10 each gives you a portfolio that is 1/√10 ≈ 32% of vol_target. To restore vol_target you need IDM ≈ √10. But if your '10 bets' are 0.7-correlated copies of the same macro view, IDM collapses to ~1.2 — there's no diversification to harvest.

Open the Excel `Discretionary` sheet and try the IDM formula with corr=0.0, 0.3, 0.7, 0.9 to feel the diversification curve.